In [ ]:
import re
import numpy
import pandas

In [ ]:
with open('./metabolic_network_v2.txt', 'r') as infile:
    data = pandas.read_csv(infile, delimiter = '\t', header = None)
data

In [ ]:
for rxn in data.values:
    if 'CPLX' in rxn[0]:
        enzyme = 'cplx(name = \'{:s}\')'.format(rxn[0].replace('-', '_'))
    else:
        enzyme = 'prot(name = \'{:s}\')'.format(rxn[0].replace('-', '_'))
        
    name = rxn[1].replace('-', '_')
    substrates = rxn[2].replace('-', '_').split(', ')
    products = rxn[3].replace('-', '_').split(', ')

    LHS = []
    RHS = []
    
    for subs in substrates:
        if subs[0].isdigit():
            subs = '_' + subs
        
        if 'PER' in subs:
            LHS.append('met(name = \'{:s}\', loc = \'per\')'.format(subs.replace('PER_', '')))
        else:
            LHS.append('met(name = \'{:s}\', loc = \'cyt\')'.format(subs))

    for prod in products:
        if prod[0].isdigit():
            prod = '_' + prod
        
        if 'PER' in prod: # inverse transport reaction
            RHS.append('met(name = \'{:s}\', loc = \'per\')'.format(prod.replace('PER_', '')))
        else:
            RHS.append('met(name = \'{:s}\', loc = \'cyt\')'.format(prod))
            
    LHS = ' + '.join(LHS)
    RHS = ' + '.join(RHS)
        
    if rxn[0] == 'spontaneous':
        Rule = 'Rule(\'{:s}\', {:s} |\n'\
               '    {:s}, \n' \
               '    Parameter(\'fwd_{:s}\', 1), \n' \
               '    Parameter(\'rvs_{:s}\', 1))' \
               .format(name, LHS, RHS, name, name)
    
    else: # need an enzyme
        Rule = 'Rule(\'{:s}\', {:s} + {:s} | \n'\
               '    {:s} + {:s}, \n' \
               '    Parameter(\'fwd_{:s}\', 1), \n' \
               '    Parameter(\'rvs_{:s}\', 0))' \
               .format(name, enzyme, LHS, enzyme, RHS, name, name)
        
    print(Rule)
    print()